In [36]:
import pandas as pd

df = pd.read_csv('combined_sale_only.csv')  



df.head()


,url,type,purpose,area,bedroom,bath,added,price,location,location_city,source,price_pkr,area_unit,area_sqft,suspected_outlier
0,https://www.zameen.com/Property/askari_askari_...,Apartment,For Sale,10 Marla,3.0,3.0,33 minutes ago,PKR 3 Crore,"Askari 11, Askari",Lahore,Zameen,30000000.0,Marla,2722.50,False
1,https://www.zameen.com/Property/gulberg_3_gulb...,Other,For Sale,2.4 Kanal,6.0,7.0,1 hour ago,PKR 15.5 Crore,"Gulberg 3 - Block M, Gulberg 3",Lahore,Zameen,155000000.0,Kanal,13068.00,False
2,https://www.zameen.com/Property/dha_phase_5_pe...,Apartment,For Sale,9.8 Marla,3.0,4.0,3 hours ago,PKR 6.95 Crore,"Penta Square By DHA Lahore, DHA Phase 5",Lahore,Zameen,69500000.0,Marla,2668.05,False
3,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,7.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block Y, DHA Phase 7",Lahore,Zameen,145000000.0,Kanal,5445.00,False
4,https://www.zameen.com/Property/dha_phase_7_dh...,House,For Sale,1 Kanal,5.0,6.0,3 hours ago,PKR 14.5 Crore,"DHA Phase 7 - Block U, DHA Phase 7",Lahore,Zameen,145000000.0,Kanal,5445.00,False


## Data Cleaning

The `price_pkr` column initially contained string values (likely due to commas and currency 
text from the original scraped data), which prevented numeric operations. This was cleaned 
by removing non-numeric characters and converting the column to a proper numeric (float) type 
before proceeding with any further steps.

## Train/Test Split

The dataset was split into training and testing sets using an 80/20 split (80% training, 20% 
testing) to evaluate model performance on unseen data. This split was performed *before* any 
feature engineering, to prevent data leakage — features derived from price averages need to be 
calculated using only the training set.

## Feature Engineering

Three main engineered features were added:

- **`avg_ppl` (average price per location)**: the mean price for each location, calculated 
  using training data only, then mapped onto both train and test sets. Locations not present 
  in training default to the overall training average.
- **`avg_ppt` (average price per property type)**: the mean price for each property type, 
  calculated the same way as `avg_ppl`.
- **`price_per_sqft`**: price divided by area in square feet, giving a normalized measure of 
  cost independent of property size.
- **`bed_bath_ratio`**: bedrooms divided by bathrooms, with a fallback of 0 for properties with 
  zero bathrooms (to avoid division errors).

Property type and city were also one-hot encoded to let the model treat each category independently.

## Model Training

A Random Forest Regressor (100 trees) was trained on the engineered feature set to predict 
property price.

## Results

- R² Score: 0.551
- Mean Absolute Error: ~14,359,029 PKR

I imported the .csv file with cleaned data frame into my Jupitor working environment using the .read_csv file to engineer the features that I deem reasonable.


I split the data using an 80/20 split.

In [37]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [38]:
print(df['price_pkr'].head(10))
print(df['price_pkr'].dtype)

0     30000000.0
1    155000000.0
2     69500000.0
3    145000000.0
4    145000000.0
5     33000000.0
6     39000000.0
7     46500000.0
8     89900000.0
9     75000000.0
Name: price_pkr, dtype: float64
float64


## **I group any number of unique loations lesser than 10 into the other category.** 

In [39]:
# Count how often each location appears in training data
location_counts = X_train['location'].value_counts()

# Locations with fewer than 10 occurrences → 'Other'
rare_locations = location_counts[location_counts < 10].index

X_train['location_grouped'] = X_train['location'].apply(
    lambda loc: 'Other' if loc in rare_locations else loc
)
X_test['location_grouped'] = X_test['location'].apply(
    lambda loc: 'Other' if loc in rare_locations else loc
)

## **I converted the dtype from string to float for some values in the price_pkr coloumn.**

In [40]:
# Check for any non-numeric junk in price_pkr
print(df['price_pkr'].apply(type).value_counts())

# If needed, clean and convert (handles commas, spaces, currency text)
df['price_pkr'] = (
    df['price_pkr']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('PKR', '', regex=False)
    .str.strip()
)
df['price_pkr'] = pd.to_numeric(df['price_pkr'], errors='coerce')

print(df['price_pkr'].isna().sum())  # how many rows failed to convert

price_pkr
<class 'float'>    3108
Name: count, dtype: int64
0


In [41]:
print(y_train.dtype)
print(y_train.head())

str
2687       Rs 11 Crore
1192    PKR 1.65 Crore
2739     Rs 4.70 Crore
2781        Rs 55 Lacs
1735     PKR 1.7 Crore
Name: price, dtype: str


In [42]:
# 1. Clean and convert price_pkr FIRST
df['price_pkr'] = (
    df['price_pkr']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('PKR', '', regex=False)
    .str.strip()
)
df['price_pkr'] = pd.to_numeric(df['price_pkr'], errors='coerce')

# 2. THEN split
from sklearn.model_selection import train_test_split

X = df.drop('price_pkr', axis=1)
y = df['price_pkr']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##  **I use the grouped_location coloumn to calculate an average price per location.**


In [43]:
train_data = X_train.copy()
train_data['price_pkr'] = y_train

location_counts = X_train['location'].value_counts()
rare_locations = location_counts[location_counts < 10].index

X_train['location_grouped'] = X_train['location'].apply(lambda loc: 'Other' if loc in rare_locations else loc)
X_test['location_grouped'] = X_test['location'].apply(lambda loc: 'Other' if loc in rare_locations else loc)

train_data['location_grouped'] = X_train['location_grouped']

avg_ppl = train_data.groupby('location_grouped')['price_pkr'].mean()  # <- note the explicit column selection
overall_avg = y_train.mean()

X_train['avg_ppl'] = X_train['location_grouped'].map(avg_ppl).fillna(overall_avg)
X_test['avg_ppl'] = X_test['location_grouped'].map(avg_ppl).fillna(overall_avg)

In [44]:
X_train[['location_grouped', 'avg_ppl']].head(10)

,location_grouped,avg_ppl
2687,Other,6.355065e+07
1192,"Airport Housing Society, Rawalpindi",2.583333e+07
2739,Other,6.355065e+07
2781,Other,6.355065e+07
1735,Other,6.355065e+07
168,"DHA Phase 7, DHA Defence",1.037939e+08
2597,Other,6.355065e+07
2829,Other,6.355065e+07
789,Other,6.355065e+07
2959,Other,6.355065e+07


In [45]:
print(df['area_sqft'].head(10))
print(df['area_sqft'].dtype)

0     2722.50
1    13068.00
2     2668.05
3     5445.00
4     5445.00
5     2722.50
6     3267.00
7     3539.25
8     5445.00
9     5445.00
Name: area_sqft, dtype: float64
float64


## **I added the "Price per square feet" feature next".**

In [46]:
X_train['price_per_sqft'] = y_train / X_train['area_sqft']
X_test['price_per_sqft'] = y_test / X_test['area_sqft']

print(X_train[['area_sqft', 'price_per_sqft']].head(10))

      area_sqft  price_per_sqft
2687   4500.000    24444.444444
1192   1361.250    12121.212121
2739   4500.000    10444.444444
2781    800.000     6875.000000
1735   1361.250    12488.521579
168    5445.000    20661.157025
2597    900.000    12222.222222
2829   4500.000    14444.444444
789    1388.475    10803.219359
2959   2178.000    39026.629936


In [47]:
print(X_train['type'].value_counts())


type
House               1360
Other                303
Apartment            301
Room                 147
Plot                 139
Residential Plot      92
Villa                 73
Shop                  21
Commercial Plot       13
Building               9
Farm House             9
Office                 8
File                   5
Penthouse              3
Upper Portion          2
Factory                1
Name: count, dtype: int64


## **I realized that all rooms were basically houses with multible rooms so I merged the room and House category.**

In [48]:
type_mapping = {'Room': 'House'}  # adjust exact strings based on what unique() shows

X_train['type'] = X_train['type'].replace(type_mapping)
X_test['type'] = X_test['type'].replace(type_mapping)

I just merged the Room type with the house as I observed that the rooms are basially properties with multiple bedrooms.


In [49]:
print(X_train['type'].value_counts())


type
House               1507
Other                303
Apartment            301
Plot                 139
Residential Plot      92
Villa                 73
Shop                  21
Commercial Plot       13
Building               9
Farm House             9
Office                 8
File                   5
Penthouse              3
Upper Portion          2
Factory                1
Name: count, dtype: int64


## **I feature engineered the average price per type to allow the model to see the price trends across different types of real estate.**

In [50]:
# No grouping — use property_type as-is, including an existing "Other" category if present
train_data['type'] = X_train['type']

avg_ppt = train_data.groupby('type')['price_pkr'].mean()
overall_avg = y_train.mean()  # fallback for any type in test but not in train

X_train['avg_ppt'] = X_train['type'].map(avg_ppt).fillna(overall_avg)
X_test['avg_ppt'] = X_test['type'].map(avg_ppt).fillna(overall_avg)

## **Checking the number of properties with 0 bathrooms in both training and testing data, to exclude them from the bed-to-bath rattio calculation.

In [51]:
print((X_train['bath'] == 0).sum())
print((X_test['bath'] == 0).sum())

248
58


## **Ensuring that the property types with zero bathrooms are actually not Houses.**

In [52]:
print(X_train[X_train['bath'] == 0]['type'].value_counts())

type
Plot                138
Residential Plot     92
Commercial Plot      12
File                  5
Factory               1
Name: count, dtype: int64


In [53]:
X_train['bed_bath_ratio'] = X_train.apply(
    lambda row: row['bedroom'] / row['bath'] if row['bath'] > 0 else 0, axis=1
)
X_test['bed_bath_ratio'] = X_test.apply(
    lambda row: row['bedroom'] / row['bath'] if row['bath'] > 0 else 0, axis=1
)

In [55]:
print(X_train.columns.tolist())

['url', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'location_city', 'source', 'area_unit', 'area_sqft', 'suspected_outlier', 'location_grouped', 'avg_ppl', 'price_per_sqft', 'avg_ppt', 'bed_bath_ratio', 'type_Apartment', 'type_Building', 'type_Commercial Plot', 'type_Factory', 'type_Farm House', 'type_File', 'type_House', 'type_Office', 'type_Other', 'type_Penthouse', 'type_Plot', 'type_Residential Plot', 'type_Shop', 'type_Upper Portion', 'type_Villa', 'location_AG Sindh Cooperative Housing Society, Karachi', 'location_ATC Villas, DHA Phase 1', 'location_AWT - Block G, Islamabad', 'location_AWT Phase 1 - Block B, Lahore', 'location_AWT Phase 2, Lahore', 'location_Abdullah Garden, Faisalabad', 'location_Abdullah Gardens, East Canal Road', 'location_Abul Hassan Isphani Road, Karachi', 'location_Adiala Road, Rawalpindi', 'location_Ahmad City, Nawabpur Road', 'location_Ahsan Colony, Multan', 'location_Air Avenue - Block L, Lahore', 'location_Airport Housing Society - Sector 3

## **Dropping all coloumns with 'str' datatype, so that the dataseet is appropriate to train a model.

In [56]:
# Drop non-numeric/unused columns, keep everything engineered + one-hot encoded
drop_cols = ['url', 'purpose','area', 'added','price', 'location_city','source','area_unit','location_grouped', 'suspected_outlier']  # adjust if these exact names differ
X_train_final = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_test_final = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns])

print(X_train_final.dtypes)  # quick check — everything should be numeric or bool now

bedroom                                           float64
bath                                              float64
area_sqft                                         float64
avg_ppl                                           float64
price_per_sqft                                    float64
                                                   ...   
location_Zameen Aurum, Gulberg 3 - Block L           bool
location_Zameen EON, NSIT City                       bool
location_Zaraj Housing Scheme, Islamabad             bool
location_Zarkon Heights, G-15                        bool
location_Zulfiqar Commercial Area, DHA Phase 8       bool
Length: 1008, dtype: object


In [57]:

print('location_grouped' in X_train.columns)

True


In [59]:
print('type' in X_train.columns)
print('location_city' in X_train.columns)

False
True


In [60]:
print(X_train.columns.tolist())

['url', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'location_city', 'source', 'area_unit', 'area_sqft', 'suspected_outlier', 'location_grouped', 'avg_ppl', 'price_per_sqft', 'avg_ppt', 'bed_bath_ratio', 'type_Apartment', 'type_Building', 'type_Commercial Plot', 'type_Factory', 'type_Farm House', 'type_File', 'type_House', 'type_Office', 'type_Other', 'type_Penthouse', 'type_Plot', 'type_Residential Plot', 'type_Shop', 'type_Upper Portion', 'type_Villa', 'location_AG Sindh Cooperative Housing Society, Karachi', 'location_ATC Villas, DHA Phase 1', 'location_AWT - Block G, Islamabad', 'location_AWT Phase 1 - Block B, Lahore', 'location_AWT Phase 2, Lahore', 'location_Abdullah Garden, Faisalabad', 'location_Abdullah Gardens, East Canal Road', 'location_Abul Hassan Isphani Road, Karachi', 'location_Adiala Road, Rawalpindi', 'location_Ahmad City, Nawabpur Road', 'location_Ahsan Colony, Multan', 'location_Air Avenue - Block L, Lahore', 'location_Airport Housing Society - Sector 3

## **Dropping all coloumns with "location_" prefix to somehow reverse the one hot-encoding of every individual location category.**

In [61]:
location_cols = [col for col in X_train.columns if col.startswith('location_')]
print(location_cols)  # sanity check before deleting
print(f"Found {len(location_cols)} location columns")

X_train = X_train.drop(columns=location_cols)
X_test = X_test.drop(columns=location_cols)

['location_city', 'location_grouped', 'location_AG Sindh Cooperative Housing Society, Karachi', 'location_ATC Villas, DHA Phase 1', 'location_AWT - Block G, Islamabad', 'location_AWT Phase 1 - Block B, Lahore', 'location_AWT Phase 2, Lahore', 'location_Abdullah Garden, Faisalabad', 'location_Abdullah Gardens, East Canal Road', 'location_Abul Hassan Isphani Road, Karachi', 'location_Adiala Road, Rawalpindi', 'location_Ahmad City, Nawabpur Road', 'location_Ahsan Colony, Multan', 'location_Air Avenue - Block L, Lahore', 'location_Airport Housing Society - Sector 3, Airport Housing Society', 'location_Airport Housing Society - Sector 4, Airport Housing Society', 'location_Airport Housing Society, Rawalpindi', 'location_Airport, Karachi', 'location_Akhtar Colony, Karachi', 'location_Al Hafeez Garden - Phase 5, Main Canal Bank Road', 'location_Al Haram Garden, Lahore', 'location_Al Jalil Garden - Block G, Lahore', 'location_Al Najaf Colony, Faisalabad', 'location_Al Noor Garden, Faisalabad',

In [62]:
print('type' in X_train.columns)
print('location_city' in X_train.columns)
print('location_grouped' in X_train.columns)

False
False
False


## **Checking if one-hot encoded coloumns of different property types exist.

In [63]:
type_cols = [col for col in X_train.columns if col.startswith('type_')]
city_cols = [col for col in X_train.columns if col.startswith('city_')]

print(f"Type columns found: {len(type_cols)}")
print(type_cols)
print(f"City columns found: {len(city_cols)}")
print(city_cols)

Type columns found: 15
['type_Apartment', 'type_Building', 'type_Commercial Plot', 'type_Factory', 'type_Farm House', 'type_File', 'type_House', 'type_Office', 'type_Other', 'type_Penthouse', 'type_Plot', 'type_Residential Plot', 'type_Shop', 'type_Upper Portion', 'type_Villa']
City columns found: 0
[]


In [ ]:
print(X_train.columns.tolist())

['url', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'location_city', 'source', 'area_unit', 'area_sqft', 'suspected_outlier', 'location_grouped', 'avg_ppl', 'price_per_sqft', 'avg_ppt', 'bed_bath_ratio', 'type_Apartment', 'type_Building', 'type_Commercial Plot', 'type_Factory', 'type_Farm House', 'type_File', 'type_House', 'type_Office', 'type_Other', 'type_Penthouse', 'type_Plot', 'type_Residential Plot', 'type_Shop', 'type_Upper Portion', 'type_Villa', 'location_AG Sindh Cooperative Housing Society, Karachi', 'location_ATC Villas, DHA Phase 1', 'location_AWT - Block G, Islamabad', 'location_AWT Phase 1 - Block B, Lahore', 'location_AWT Phase 2, Lahore', 'location_Abdullah Garden, Faisalabad', 'location_Abdullah Gardens, East Canal Road', 'location_Abul Hassan Isphani Road, Karachi', 'location_Adiala Road, Rawalpindi', 'location_Ahmad City, Nawabpur Road', 'location_Ahsan Colony, Multan', 'location_Air Avenue - Block L, Lahore', 'location_Airport Housing Society - Sector 3

In [64]:
location_cols = [col for col in X_train.columns if col.startswith('location_')]
print(f"Dropping {len(location_cols)} location columns:")
print(location_cols)

X_train = X_train.drop(columns=location_cols)
X_test = X_test.drop(columns=location_cols)

Dropping 0 location columns:
[]


In [65]:
print('city' in X_train.columns)
city_cols = [col for col in X_train.columns if col.startswith('city_')]
print(f"city_ columns found: {len(city_cols)}")
print(city_cols)

False
city_ columns found: 0
[]


In [66]:
# Recover location_city from the original df using matching row indices
X_train['location_city'] = df.loc[X_train.index, 'location_city']
X_test['location_city'] = df.loc[X_test.index, 'location_city']

# Confirm it's back and looks right
print(X_train['location_city'].head())
print(X_train['location_city'].isna().sum())  # should be 0

2687       Karachi
1192    Rawalpindi
2739       Karachi
2781       Karachi
1735    Faisalabad
Name: location_city, dtype: str
0


In [67]:
X_train = pd.get_dummies(X_train, columns=['location_city'], prefix='city')
X_test = pd.get_dummies(X_test, columns=['location_city'], prefix='city')
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [68]:
print(df.columns.tolist())

['url', 'type', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'location', 'location_city', 'source', 'price_pkr', 'area_unit', 'area_sqft', 'suspected_outlier']


In [69]:
print(X_train.columns.tolist())

['url', 'purpose', 'area', 'bedroom', 'bath', 'added', 'price', 'source', 'area_unit', 'area_sqft', 'suspected_outlier', 'avg_ppl', 'price_per_sqft', 'avg_ppt', 'bed_bath_ratio', 'type_Apartment', 'type_Building', 'type_Commercial Plot', 'type_Factory', 'type_Farm House', 'type_File', 'type_House', 'type_Office', 'type_Other', 'type_Penthouse', 'type_Plot', 'type_Residential Plot', 'type_Shop', 'type_Upper Portion', 'type_Villa', 'city_Faisalabad', 'city_Islamabad', 'city_Karachi', 'city_Lahore', 'city_Multan', 'city_Rawalpindi']


## **Checking the dtype for the final list of coloumns in the training and testing dataset.**


In [70]:
print(X_train.dtypes.value_counts())

bool       22
str         7
float64     7
Name: count, dtype: int64


In [71]:
string_cols = X_train.select_dtypes(include='object').columns.tolist()
print(string_cols)

['url', 'purpose', 'area', 'added', 'price', 'source', 'area_unit']


C:\Users\Liza\AppData\Local\Temp\ipykernel_29840\4171208904.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = X_train.select_dtypes(include='object').columns.tolist()


## **Dropping String coloumns since only numeric dtypes are included for model training.**


In [72]:
X_train = X_train.drop(columns=string_cols)
X_test = X_test.drop(columns=string_cols)

In [73]:
print(X_train.dtypes.value_counts())

bool       22
float64     7
Name: count, dtype: int64


## **Training the model**

In [74]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:,.0f} PKR")
print(f"R² Score: {r2:.3f}")

Mean Absolute Error: 14,359,029 PKR
R² Score: 0.551


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# R² Score (variance explained)
r2 = r2_score(y_test, y_pred)

# Mean Absolute Error
mae = mean_absolute_error(y_test, y_pred)

# Root Mean Squared Error
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Mean Absolute Percentage Error
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

# Median Absolute Percentage Error
median_ape = np.median(np.abs((y_test - y_pred) / y_test)) * 100

print(f"R² Score (Variance Explained): {r2:.3f}")
print(f"Mean Absolute Error: {mae:,.0f} PKR")
print(f"Root Mean Squared Error: {rmse:,.0f} PKR")
print(f"Mean Absolute Percentage Error: {mape:.2f}%")
print(f"Median Absolute Percentage Error: {median_ape:.2f}%")


R² Score (Variance Explained): 0.551
Mean Absolute Error: 14,359,029 PKR
Root Mean Squared Error: 180,023,946 PKR
Mean Absolute Percentage Error: 11.43%
Median Absolute Percentage Error: 0.83%
